In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import time
from sklearn.metrics import accuracy_score

from src.data import load_fashion_mnist

from src.preprocessing import (
    flatten_images,
    scale_pixels,
    create_validation_split,
    fit_pca,
    transform_pca
)

from src.models import create_logistic_regression

In [4]:
X_train, y_train, X_test, y_test = load_fashion_mnist()

X_train_flat = flatten_images(X_train)
X_test_flat = flatten_images(X_test)

X_train_scaled = scale_pixels(X_train_flat)
X_test_scaled = scale_pixels(X_test_flat)

X_fit, X_val, y_fit, y_val = create_validation_split(
    X_train_scaled,
    y_train
)
PCA_COMPONENTS = 187

pca, X_fit_pca = fit_pca(
    X_fit,
    PCA_COMPONENTS
)
X_val_pca = transform_pca(
    pca,
    X_val
)

X_test_pca = transform_pca(
    pca,
    X_test_scaled
)
print("Fit:", X_fit_pca.shape)
print("Validation:", X_val_pca.shape)
print("Test:", X_test_pca.shape)

Fit: (48000, 187)
Validation: (12000, 187)
Test: (10000, 187)


In [5]:
logistic_model = create_logistic_regression()
start_time = time.perf_counter()

logistic_model.fit(
    X_fit_pca,
    y_fit
)

fit_time = time.perf_counter() - start_time

In [6]:
y_val_pred = logistic_model.predict(
    X_val_pca
)

val_accuracy = accuracy_score(
    y_val,
    y_val_pred
)

print(
    f"Validation accuracy: {val_accuracy:.4f}"
)

Validation accuracy: 0.8642


In [7]:
# Checkpoint-1 official test prediction

start_time = time.perf_counter()

y_test_pred = logistic_model.predict(X_test_pca)

predict_time = time.perf_counter() - start_time

test_accuracy = accuracy_score(
    y_test,
    y_test_pred
)

print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Fit time: {fit_time:.4f} seconds")
print(f"Predict time: {predict_time:.4f} seconds")

Test accuracy: 0.8451
Fit time: 42.8113 seconds
Predict time: 0.0064 seconds


## Checkpoint 1 — Logistic Regression Baseline

A Logistic Regression classifier was selected as the initial
baseline.

Images were flattened, normalized to [0, 1], and transformed
using PCA fitted exclusively on the model-fitting portion of
the official training set.

The primary PCA representation retains at least 95% cumulative
explained variance, corresponding to 187 principal components.

The Logistic Regression model was trained on the PCA-transformed
model-fitting subset.

**Validation accuracy:**  0.8642

**Official test accuracy:** 0.8451 (84.51%)

**Fit time:** 42.8113 seconds

**Official test prediction time:** 0.0064 seconds

This result serves as the initial linear baseline against which
the later KNN, SVM, Random Forest, and MLP classifiers will be
compared.